# 🛠️ Chapter 05-01b: Hands-On — Chat with Your PDF in 50 Lines

---

## Build a Complete RAG Pipeline! 📄➡️🤖

In this notebook, you'll build end-to-end RAG from scratch.

By the end:
- ✅ Loaded documents (PDF, web, text)
- ✅ Split text into smart chunks
- ✅ Stored in a vector database
- ✅ Built a RAG chain with LangChain
- ✅ Chat with your data!

Let's build! 🚀

---

## 📦 Setup

In [ ]:
# !pip install langchain-openai langchain-community langchain-core chromadb python-dotenv bs4

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ Ready!" if os.getenv("OPENAI_API_KEY") else "❌ Set OPENAI_API_KEY in .env")

---

## 🧪 Step 1: Load Documents

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

# Load a webpage (using a Wikipedia article as example)
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Retrieval-augmented_generation")
docs = loader.load()

print(f"📄 Loaded {len(docs)} document(s)")
print(f"📏 Total characters: {len(docs[0].page_content):,}")
print(f"📋 Metadata: {docs[0].metadata}")
print(f"\n📝 Preview (first 300 chars):")
print(docs[0].page_content[:300])

---

## 🧪 Step 2: Split into Chunks

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # max characters per chunk
    chunk_overlap=200,     # overlap between chunks
    separators=["\n\n", "\n", ". ", " ", ""]  # split priority
)

chunks = text_splitter.split_documents(docs)

print(f"✂️ Split into {len(chunks)} chunks")
print(f"📏 Avg chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")

# Show first 3 chunks
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:150] + "...")

---

## 🧪 Step 3: Embed & Store in ChromaDB

In [ ]:
from langchain_community.vectorstores import Chroma

# One line! LangChain handles embedding + storing
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_demo"
)

# Create a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"✅ Vector store created with {len(chunks)} chunks")
print(f"   Retriever will return top 4 chunks per query")

In [ ]:
# Test retrieval
test_results = retriever.invoke("What is RAG?")

print(f"🔍 Retrieved {len(test_results)} chunks for 'What is RAG?'\n")
for i, doc in enumerate(test_results, 1):
    print(f"Chunk {i}: {doc.page_content[:100]}...")
    print()

---

## 🧪 Step 4: Build the RAG Chain

In [ ]:
# The RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. Answer the question based ONLY on the provided context.
If the context doesn't contain the answer, say "I don't have enough information to answer that."
Always cite which part of the context you used."""),
    ("human", """Context:
{context}

Question: {question}

Answer:""")
])

# Helper: format retrieved docs into a single string
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain with LCEL
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain built!")
print("   Flow: Question → Retrieve → Format → Prompt → LLM → Answer")

In [ ]:
# Ask questions!
questions = [
    "What is Retrieval-Augmented Generation?",
    "Who introduced RAG and when?",
    "What are the main components of a RAG system?",
]

for q in questions:
    print(f"❓ {q}")
    answer = rag_chain.invoke(q)
    print(f"💬 {answer}")
    print("═" * 70 + "\n")

In [ ]:
# Test hallucination resistance
out_of_scope = rag_chain.invoke("What is the capital of Mars?")
print(f"❓ What is the capital of Mars?")
print(f"💬 {out_of_scope}")
print("\n💡 The model should say it doesn't know — that's the anti-hallucination prompt working!")

---

## 🚀 The 50-Line Version

In [ ]:
# Chat with ANY URL in ~50 lines!

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Load
docs = WebBaseLoader("https://en.wikipedia.org/wiki/Large_language_model").load()

# 2. Split
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)

# 3. Embed & Store
store = Chroma.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small"))

# 4. Retrieve & Generate
chain = (
    {"context": store.as_retriever(search_kwargs={"k": 4}) | (lambda docs: "\n\n".join(d.page_content for d in docs)),
     "question": RunnablePassthrough()}
    | ChatPromptTemplate.from_messages([
        ("system", "Answer based ONLY on the context. If unsure, say 'I don't know.'"),
        ("human", "Context: {context}\n\nQuestion: {question}")])
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
)

# Chat!
print(chain.invoke("What are large language models?"))

---

## ✍️ Practice: RAG with Custom Text

**Challenge:** Build a RAG system over a custom knowledge base.

In [ ]:
# YOUR CODE HERE

company_docs = [
    "Our company was founded in 2020 by Jane Smith and John Doe.",
    "We offer three products: DataPro ($99/mo), DataPro Plus ($199/mo), and Enterprise (custom pricing).",
    "Our headquarters is in San Francisco, with offices in London and Tokyo.",
    "The engineering team has 50 members across 3 time zones.",
    "Our annual revenue in 2024 was $15M, up 40% from 2023.",
    "We use Python, React, and PostgreSQL as our main tech stack.",
    "Customer support is available 24/7 via email at support@company.com.",
    "Our main competitors are DataCorp and AnalyticsPro.",
]

# TODO: Create a Chroma vector store from company_docs
# TODO: Build a RAG chain
# TODO: Test with: "How much does DataPro cost?", "Who founded the company?"

<details>
<summary>💡 Click to see solution</summary>

```python
company_store = Chroma.from_texts(
    texts=company_docs,
    embedding=embeddings
)

company_chain = (
    {"context": company_store.as_retriever(search_kwargs={"k": 3}) | format_docs,
     "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

for q in ["How much does DataPro cost?", "Who founded the company?", "What's the weather?"]:
    print(f"Q: {q}")
    print(f"A: {company_chain.invoke(q)}\n")
```

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Load → Split → Embed → Store → Retrieve → Generate.           │
├─────────────────────────────────────────────────────────────────────┤
│  📌 RecursiveCharacterTextSplitter with overlap is the starting pt.│
├─────────────────────────────────────────────────────────────────────┤
│  📌 Chroma.from_documents() handles embed+store in one call.       │
├─────────────────────────────────────────────────────────────────────┤
│  📌 LCEL makes the RAG chain clean: retriever | prompt | llm.      │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Anti-hallucination prompt: "say I don't know if unsure."       │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Advanced RAG Techniques!

Chunking strategies, hybrid search, re-ranking, HyDE, and more.

---

### 🎉 You Built RAG!
From document to chatbot. The #1 skill in GenAI engineering. 🚀